# Group Eta — WRI + OWID + World Bank Data Integration

This notebook integrates three independent data sources for the project:

1. **Global Power Plant Database (WRI)**
   - plant-level characteristics
   - installed generation capacity
   - primary fuel and location

2. **Our World in Data (OWID) CO₂ Dataset**
   - national emissions and carbon indicators

3. **World Bank Indicators API**
   - GDP
   - population
   - GDP per capita
   - retrieved programmatically as JSON

The final dataset preserves the natural plant-level grain:

**One row = one real power plant**

Country-level information from OWID and the World Bank is joined to the plant records using ISO-3 country codes.

The notebook produces one final output:

**`eta_merged_2019.csv`**

## 1. Import Libraries and Load Source Data

In [ ]:
import pandas as pd
import numpy as np
import requests

df_power = pd.read_csv(
    'global_power_plant_database.csv',
    low_memory=False
)

df_co2 = pd.read_csv(
    'owid-co2-data-1.csv',
    low_memory=False
)

print("Power plant dataset:", df_power.shape)
print("CO2 dataset:", df_co2.shape)

Power plant dataset: (34936, 36)
CO2 dataset: (50411, 79)


### 1.1 Source Coverage Check

In [ ]:
original_rows = len(df_power)

original_countries = (
    df_power['country']
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
)

missing_plant_ids = df_power['gppd_idnr'].isna().sum()

duplicate_plant_ids = (
    df_power.loc[
        df_power['gppd_idnr'].notna(),
        'gppd_idnr'
    ]
    .duplicated()
    .sum()
)

philippine_rows = (
    original_countries.eq('PHL')
).sum()

print("Original power plant rows:", original_rows)
print("Unique plant IDs:", df_power['gppd_idnr'].nunique())
print("Missing plant IDs:", missing_plant_ids)
print("Duplicate plant IDs:", duplicate_plant_ids)
print("Countries in original dataset:", original_countries.nunique())
print("Philippines present?:", 'PHL' in set(original_countries))
print("Philippine plant rows:", philippine_rows)

Original power plant rows: 34936
Unique plant IDs: 34936
Missing plant IDs: 0
Duplicate plant IDs: 0
Countries in original dataset: 167
Philippines present?: True
Philippine plant rows: 123


## 2. Clean Global Power Plant Database

The Global Power Plant Database is used as the base table for the final merged dataset.

The plant-level grain is preserved throughout the pipeline:

**One row = one real power plant**

Rows are not removed based on missing `generation_gwh` values. The main dataset is based on plant identity, installed capacity, fuel type, geographic information, and other real plant attributes available in the source.

In [ ]:
# Preserve the complete plant-level source table
df_plants = df_power.copy()

# Standardize country ISO-3 codes
df_plants['country'] = (
    df_plants['country']
    .astype(str)
    .str.strip()
    .str.upper()
)

# Standardize primary fuel labels
df_plants['primary_fuel'] = (
    df_plants['primary_fuel']
    .astype(str)
    .str.strip()
)

# Ensure installed capacity is numeric
df_plants['capacity_mw'] = pd.to_numeric(
    df_plants['capacity_mw'],
    errors='coerce'
)

# Keep records with the core fields required for the plant-level dataset
df_plants = df_plants.dropna(
    subset=[
        'country',
        'gppd_idnr',
        'primary_fuel',
        'capacity_mw'
    ]
).copy()

# Installed capacity should be positive
df_plants = df_plants[
    df_plants['capacity_mw'] > 0
].copy()

print("Clean plant-level rows:", len(df_plants))
print("Unique plant IDs:", df_plants['gppd_idnr'].nunique())
print("Countries:", df_plants['country'].nunique())
print(
    "Philippines present?:",
    'PHL' in set(df_plants['country'])
)

df_plants.head()

Clean plant-level rows: 34936
Unique plant IDs: 34936
Countries: 167
Philippines present?: True


,country,country_long,name,gppd_idnr,capacity_mw,latitude,longitude,primary_fuel,other_fuel1,other_fuel2,...,estimated_generation_gwh_2013,estimated_generation_gwh_2014,estimated_generation_gwh_2015,estimated_generation_gwh_2016,estimated_generation_gwh_2017,estimated_generation_note_2013,estimated_generation_note_2014,estimated_generation_note_2015,estimated_generation_note_2016,estimated_generation_note_2017
0,AFG,Afghanistan,Kajaki Hydroelectric Power Plant Afghanistan,GEODB0040538,33.0,32.322,65.1190,Hydro,NaN,NaN,...,123.77,162.90,97.39,137.76,119.50,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1
1,AFG,Afghanistan,Kandahar DOG,WKS0070144,10.0,31.670,65.7950,Solar,NaN,NaN,...,18.43,17.48,18.25,17.70,18.29,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE
2,AFG,Afghanistan,Kandahar JOL,WKS0071196,10.0,31.623,65.7920,Solar,NaN,NaN,...,18.64,17.58,19.10,17.62,18.72,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE
3,AFG,Afghanistan,Mahipar Hydroelectric Power Plant Afghanistan,GEODB0040541,66.0,34.556,69.4787,Hydro,NaN,NaN,...,225.06,203.55,146.90,230.18,174.91,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1
4,AFG,Afghanistan,Naghlu Dam Hydroelectric Power Plant Afghanistan,GEODB0040534,100.0,34.641,69.7170,Hydro,NaN,NaN,...,406.16,357.22,270.99,395.38,350.80,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1


## 3. Clean OWID 2019 Emissions Snapshot

The OWID dataset is reduced to a single 2019 observation per ISO-3 country code.

OWID is used primarily for national emissions and carbon indicators.

GDP and population are retained temporarily for source comparison but are excluded from the final OWID merge because the World Bank will serve as the primary source for economic and demographic indicators.

The final relationship remains:

**many power plants → one OWID country record**

In [ ]:
df_co2_2019 = df_co2[
    df_co2['iso_code'].notna()
].copy()

# Keep standard ISO-3 country codes only
df_co2_2019 = df_co2_2019[
    df_co2_2019['iso_code'].str.match(
        r'^[A-Z]{3}$',
        na=False
    )
].copy()

# Keep only the 2019 country snapshot
df_co2_2019 = df_co2_2019[
    df_co2_2019['year'].eq(2019)
].copy()

# Rename OWID country name to avoid conflict
# with the GPPD country code column
df_co2_2019 = df_co2_2019.rename(
    columns={
        'country': 'owid_country'
    }
)

# Keep OWID GDP and population separately
# for optional source-comparison validation later.
owid_economic_validation_2019 = (
    df_co2_2019[
        [
            'iso_code',
            'population',
            'gdp'
        ]
    ]
    .rename(
        columns={
            'population': 'owid_population_compare',
            'gdp': 'owid_gdp_compare'
        }
    )
    .copy()
)

# Remove GDP and population from the OWID table
# used in the final integration.
# World Bank will become the primary source
# for these economic/demographic variables.
df_co2_2019 = df_co2_2019.drop(
    columns=[
        'population',
        'gdp'
    ]
)

duplicate_iso_2019 = df_co2_2019.duplicated(
    subset=['iso_code']
).sum()

print("2019 OWID rows:", len(df_co2_2019))

print(
    "Unique ISO-3 countries:",
    df_co2_2019['iso_code'].nunique()
)

print(
    "Duplicate ISO-3 codes:",
    duplicate_iso_2019
)

print(
    "OWID population in final OWID table?:",
    'population' in df_co2_2019.columns
)

print(
    "OWID GDP in final OWID table?:",
    'gdp' in df_co2_2019.columns
)

display(
    df_co2_2019[
        [
            col for col in [
                'owid_country',
                'iso_code',
                'year',
                'co2',
                'co2_per_capita',
                'co2_per_gdp',
                'cumulative_co2',
                'share_global_co2'
            ]
            if col in df_co2_2019.columns
        ]
    ].head()
)

2019 OWID rows: 218
Unique ISO-3 countries: 218
Duplicate ISO-3 codes: 0
OWID population in final OWID table?: False
OWID GDP in final OWID table?: False


,owid_country,iso_code,year,co2,co2_per_capita,co2_per_gdp,cumulative_co2,share_global_co2
269,Afghanistan,AFG,2019,10.400,0.275,0.142,190.136,0.028
894,Albania,ALB,2019,4.941,1.712,0.148,284.576,0.013
1069,Algeria,DZA,2019,182.425,4.214,0.309,4529.565,0.492
1344,Andorra,AND,2019,0.491,6.419,NaN,14.597,0.001
1519,Angola,AGO,2019,22.210,0.686,0.138,638.722,0.060


## 4. Country-Level Feature Engineering

Country-level installed-capacity metrics are calculated from the plant-level GPPD records.

These grouped tables are temporary feature-engineering tables only.

They will later be merged back onto the individual plant records so that the final dataset remains at the plant level.

In [ ]:
# Total installed capacity and plant count per country
country_totals = (
    df_plants
    .groupby(
        'country',
        as_index=False
    )
    .agg(
        total_plant_count=(
            'gppd_idnr',
            'nunique'
        ),
        total_installed_capacity_mw=(
            'capacity_mw',
            'sum'
        )
    )
)

# Installed capacity by country and primary fuel
country_fuel_capacity = (
    df_plants
    .groupby(
        ['country', 'primary_fuel'],
        as_index=False
    )
    .agg(
        plant_count=(
            'gppd_idnr',
            'nunique'
        ),
        capacity_mw=(
            'capacity_mw',
            'sum'
        )
    )
)

# Add total country installed capacity
country_fuel_capacity = country_fuel_capacity.merge(
    country_totals[
        [
            'country',
            'total_installed_capacity_mw'
        ]
    ],
    on='country',
    how='left',
    validate='many_to_one'
)

# Installed-capacity share of each primary fuel
country_fuel_capacity['capacity_share_pct'] = (
    country_fuel_capacity['capacity_mw']
    / country_fuel_capacity['total_installed_capacity_mw']
    * 100
)

print(
    "Country-level rows:",
    len(country_totals)
)

print(
    "Country-fuel combinations:",
    len(country_fuel_capacity)
)

print(
    "Countries represented:",
    country_fuel_capacity['country'].nunique()
)

display(
    country_fuel_capacity.head(20)
)

Country-level rows: 167
Country-fuel combinations: 698
Countries represented: 167


,country,primary_fuel,plant_count,capacity_mw,total_installed_capacity_mw,capacity_share_pct
0,AFG,Gas,1,42.000,300.550,13.974380
1,AFG,Hydro,6,238.550,300.550,79.371153
2,AFG,Solar,2,20.000,300.550,6.654467
3,AGO,Gas,3,163.680,1071.180,15.280345
4,AGO,Hydro,5,770.600,1071.180,71.939357
5,AGO,Oil,6,136.900,1071.180,12.780298
6,ALB,Hydro,7,1431.000,1529.000,93.590582
7,ALB,Other,1,98.000,1529.000,6.409418
8,ARE,Gas,24,29487.000,30327.000,97.230191
9,ARE,Solar,6,840.000,30327.000,2.769809


### 4.1 Build Country Installed-Capacity Features

A temporary one-row-per-country feature table is created.

Fuel-specific installed capacity and installed-capacity shares are pivoted into separate columns.

Examples:

- `Coal_capacity_mw`
- `Coal_share_pct`
- `Gas_capacity_mw`
- `Gas_share_pct`
- `Hydro_capacity_mw`
- `Hydro_share_pct`

These values describe **installed capacity**, not actual electricity generation.

This temporary country-level feature table will later be merged back onto every real power plant.

In [ ]:
# Fuel-specific installed capacity
capacity_pivot = (
    country_fuel_capacity
    .pivot(
        index='country',
        columns='primary_fuel',
        values='capacity_mw'
    )
    .fillna(0)
)

capacity_pivot.columns = [
    f"{fuel}_capacity_mw"
    for fuel in capacity_pivot.columns
]

capacity_pivot = capacity_pivot.reset_index()


# Fuel-specific installed-capacity share
share_pivot = (
    country_fuel_capacity
    .pivot(
        index='country',
        columns='primary_fuel',
        values='capacity_share_pct'
    )
    .fillna(0)
)

share_pivot.columns = [
    f"{fuel}_share_pct"
    for fuel in share_pivot.columns
]

share_pivot = share_pivot.reset_index()


# Combine all country-level capacity features
country_capacity_profile = (
    country_totals
    .merge(
        capacity_pivot,
        on='country',
        how='left',
        validate='one_to_one'
    )
    .merge(
        share_pivot,
        on='country',
        how='left',
        validate='one_to_one'
    )
)

print(
    "Country capacity profile shape:",
    country_capacity_profile.shape
)

print(
    "Countries in capacity profile:",
    country_capacity_profile['country'].nunique()
)

display(
    country_capacity_profile.head()
)

Country capacity profile shape: (167, 33)
Countries in capacity profile: 167


,country,total_plant_count,total_installed_capacity_mw,Biomass_capacity_mw,Coal_capacity_mw,Cogeneration_capacity_mw,Gas_capacity_mw,Geothermal_capacity_mw,Hydro_capacity_mw,Nuclear_capacity_mw,...,Hydro_share_pct,Nuclear_share_pct,Oil_share_pct,Other_share_pct,Petcoke_share_pct,Solar_share_pct,Storage_share_pct,Waste_share_pct,Wave and Tidal_share_pct,Wind_share_pct
0,AFG,9,300.550,0.0,0.0,0.0,42.00,0.0,238.55,0.0,...,79.371153,0.000000,0.000000,0.000000,0.0,6.654467,0.0,0.0,0.0,0.000000
1,AGO,14,1071.180,0.0,0.0,0.0,163.68,0.0,770.60,0.0,...,71.939357,0.000000,12.780298,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000
2,ALB,8,1529.000,0.0,0.0,0.0,0.00,0.0,1431.00,0.0,...,93.590582,0.000000,0.000000,6.409418,0.0,0.000000,0.0,0.0,0.0,0.000000
3,ARE,30,30327.000,0.0,0.0,0.0,29487.00,0.0,0.00,0.0,...,0.000000,0.000000,0.000000,0.000000,0.0,2.769809,0.0,0.0,0.0,0.000000
4,ARG,236,32913.079,0.0,4857.4,0.0,13041.84,0.0,9999.71,1763.0,...,30.382177,5.356533,3.643807,4.024115,0.0,1.566854,0.0,0.0,0.0,0.643149


### 4.2 Fossil and Renewable Installed-Capacity Features

Power plant fuels are grouped conservatively into fossil, renewable, and other/unclassified categories.

Fossil fuels:
- Coal
- Gas
- Oil
- Petcoke

Renewable fuels:
- Biomass
- Geothermal
- Hydro
- Solar
- Wind
- Waste
- Wave and Tidal

Nuclear, storage, cogeneration, and unspecified/other fuels are kept under `other_or_unclassified` rather than being incorrectly classified as fossil or renewable.

The resulting variables represent **installed-capacity shares**, not electricity-generation shares.

In [ ]:
fossil_fuels = {
    'Coal',
    'Gas',
    'Oil',
    'Petcoke'
}

renewable_fuels = {
    'Biomass',
    'Geothermal',
    'Hydro',
    'Solar',
    'Wind',
    'Waste',
    'Wave and Tidal'
}

df_capacity_groups = df_plants[
    [
        'country',
        'gppd_idnr',
        'capacity_mw',
        'primary_fuel'
    ]
].copy()

df_capacity_groups['capacity_group'] = np.select(
    [
        df_capacity_groups['primary_fuel'].isin(
            fossil_fuels
        ),
        df_capacity_groups['primary_fuel'].isin(
            renewable_fuels
        )
    ],
    [
        'fossil',
        'renewable'
    ],
    default='other_or_unclassified'
)

country_group_capacity = (
    df_capacity_groups
    .groupby(
        ['country', 'capacity_group'],
        as_index=False
    )
    .agg(
        capacity_mw=(
            'capacity_mw',
            'sum'
        )
    )
)

country_group_profile = (
    country_group_capacity
    .pivot(
        index='country',
        columns='capacity_group',
        values='capacity_mw'
    )
    .fillna(0)
    .reset_index()
)

# Make sure all three group columns exist
for group in [
    'fossil',
    'renewable',
    'other_or_unclassified'
]:
    if group not in country_group_profile.columns:
        country_group_profile[group] = 0

country_group_profile = (
    country_group_profile
    .rename(
        columns={
            'fossil':
                'fossil_capacity_mw',
            'renewable':
                'renewable_capacity_mw',
            'other_or_unclassified':
                'other_or_unclassified_capacity_mw'
        }
    )
)

country_group_profile = country_group_profile.merge(
    country_totals[
        [
            'country',
            'total_installed_capacity_mw'
        ]
    ],
    on='country',
    how='left',
    validate='one_to_one'
)

country_group_profile['fossil_share_pct'] = (
    country_group_profile['fossil_capacity_mw']
    / country_group_profile[
        'total_installed_capacity_mw'
    ]
    * 100
)

country_group_profile['renewable_share_pct'] = (
    country_group_profile['renewable_capacity_mw']
    / country_group_profile[
        'total_installed_capacity_mw'
    ]
    * 100
)

country_group_profile[
    'other_or_unclassified_share_pct'
] = (
    country_group_profile[
        'other_or_unclassified_capacity_mw'
    ]
    / country_group_profile[
        'total_installed_capacity_mw'
    ]
    * 100
)

# total_installed_capacity_mw already exists
# in country_capacity_profile, so remove the
# duplicate before combining the feature tables
country_group_profile = country_group_profile.drop(
    columns=['total_installed_capacity_mw']
)

country_capacity_features = (
    country_capacity_profile
    .merge(
        country_group_profile,
        on='country',
        how='left',
        validate='one_to_one'
    )
)

print(
    "Final country feature table:",
    country_capacity_features.shape
)

display(
    country_capacity_features.head()
)

Final country feature table: (167, 39)


,country,total_plant_count,total_installed_capacity_mw,Biomass_capacity_mw,Coal_capacity_mw,Cogeneration_capacity_mw,Gas_capacity_mw,Geothermal_capacity_mw,Hydro_capacity_mw,Nuclear_capacity_mw,...,Storage_share_pct,Waste_share_pct,Wave and Tidal_share_pct,Wind_share_pct,fossil_capacity_mw,other_or_unclassified_capacity_mw,renewable_capacity_mw,fossil_share_pct,renewable_share_pct,other_or_unclassified_share_pct
0,AFG,9,300.550,0.0,0.0,0.0,42.00,0.0,238.55,0.0,...,0.0,0.0,0.0,0.000000,42.000,0.00,258.55,13.974380,86.025620,0.000000
1,AGO,14,1071.180,0.0,0.0,0.0,163.68,0.0,770.60,0.0,...,0.0,0.0,0.0,0.000000,300.580,0.00,770.60,28.060643,71.939357,0.000000
2,ALB,8,1529.000,0.0,0.0,0.0,0.00,0.0,1431.00,0.0,...,0.0,0.0,0.0,0.000000,0.000,98.00,1431.00,0.000000,93.590582,6.409418
3,ARE,30,30327.000,0.0,0.0,0.0,29487.00,0.0,0.00,0.0,...,0.0,0.0,0.0,0.000000,29487.000,0.00,840.00,97.230191,2.769809,0.000000
4,ARG,236,32913.079,0.0,4857.4,0.0,13041.84,0.0,9999.71,1763.0,...,0.0,0.0,0.0,0.643149,19098.529,3087.46,10727.09,58.027172,32.592180,9.380648


## 5. Retrieve World Bank 2019 Indicators

World Bank World Development Indicators are retrieved programmatically through the official World Bank Indicators REST API.

The API response is requested in JSON format and normalized into a country-level pandas DataFrame.

The selected 2019 indicators are:

- `NY.GDP.MKTP.CD` — GDP (current US$)
- `SP.POP.TOTL` — Population, total
- `NY.GDP.PCAP.CD` — GDP per capita (current US$)

World Bank serves as the primary source for economic and demographic indicators in the final dataset.

No missing World Bank values are replaced with zero.

### 5.1 World Bank API Configuration

In [ ]:
WB_BASE_URL = (
    'https://api.worldbank.org/v2'
)

WB_YEAR = 2019

WB_INDICATORS = {
    'NY.GDP.MKTP.CD':
        'wb_gdp_current_usd',

    'SP.POP.TOTL':
        'wb_population',

    'NY.GDP.PCAP.CD':
        'wb_gdp_per_capita_current_usd'
}

print("World Bank year:", WB_YEAR)

print("\nWorld Bank indicators:")
for indicator, column_name in WB_INDICATORS.items():
    print(
        f"{indicator} -> {column_name}"
    )

World Bank year: 2019

World Bank indicators:
NY.GDP.MKTP.CD -> wb_gdp_current_usd
SP.POP.TOTL -> wb_population
NY.GDP.PCAP.CD -> wb_gdp_per_capita_current_usd


### 5.2 Retrieve and Normalize JSON Responses

Each World Bank indicator is retrieved separately.

The retrieval function:

- requests JSON from the official V2 API,
- checks the HTTP response,
- handles pagination,
- validates the JSON response structure,
- retains ISO-3 country codes,
- keeps the 2019 observation,
- and prevents duplicate country keys.

In [ ]:
def fetch_world_bank_indicator(
    indicator_code,
    output_column,
    year=2019
):
    url = (
        f"{WB_BASE_URL}"
        f"/country/all/indicator/"
        f"{indicator_code}"
    )

    page = 1
    all_records = []

    while True:
        params = {
            'format': 'json',
            'date': str(year),
            'page': page,
            'per_page': 200
        }

        try:
            response = requests.get(
                url,
                params=params,
                timeout=30
            )

            response.raise_for_status()

        except requests.RequestException as error:
            raise RuntimeError(
                f"World Bank API request failed "
                f"for {indicator_code}: {error}"
            )

        try:
            payload = response.json()

        except ValueError as error:
            raise RuntimeError(
                f"Invalid JSON response for "
                f"{indicator_code}"
            ) from error

        if (
            not isinstance(payload, list)
            or len(payload) < 2
            or not isinstance(payload[0], dict)
        ):
            raise RuntimeError(
                f"Unexpected World Bank API "
                f"response for {indicator_code}"
            )

        metadata = payload[0]
        records = payload[1]

        if records is None:
            records = []

        if not isinstance(records, list):
            raise RuntimeError(
                f"Invalid record structure for "
                f"{indicator_code}"
            )

        all_records.extend(records)

        total_pages = int(
            metadata.get('pages', 1)
        )

        if page >= total_pages:
            break

        page += 1

    if len(all_records) == 0:
        raise RuntimeError(
            f"No World Bank records returned "
            f"for {indicator_code}"
        )

    rows = []

    for record in all_records:

        country_info = (
            record.get('country') or {}
        )

        rows.append(
            {
                'iso_code':
                    record.get(
                        'countryiso3code'
                    ),

                'wb_country':
                    country_info.get(
                        'value'
                    ),

                'wb_year':
                    pd.to_numeric(
                        record.get('date'),
                        errors='coerce'
                    ),

                output_column:
                    pd.to_numeric(
                        record.get('value'),
                        errors='coerce'
                    )
            }
        )

    indicator_df = pd.DataFrame(
        rows
    )

    # Keep valid three-letter country codes
    indicator_df = indicator_df[
        indicator_df['iso_code']
        .fillna('')
        .str.match(
            r'^[A-Z]{3}$'
        )
    ].copy()

    # Keep only requested year
    indicator_df = indicator_df[
        indicator_df['wb_year']
        .eq(year)
    ].copy()

    duplicate_iso = (
        indicator_df
        .duplicated(
            subset=['iso_code']
        )
        .sum()
    )

    if duplicate_iso > 0:
        raise RuntimeError(
            f"{indicator_code} returned "
            f"{duplicate_iso} duplicate "
            f"ISO-3 country codes."
        )

    print(
        f"{indicator_code}: "
        f"{len(indicator_df)} country rows, "
        f"{indicator_df[output_column].notna().sum()} "
        f"non-null values"
    )

    return indicator_df

### 5.3 Prepare World Bank 2019 Country Table

In [ ]:
wb_indicator_tables = {}

for indicator_code, output_column in (
    WB_INDICATORS.items()
):
    wb_indicator_tables[
        indicator_code
    ] = fetch_world_bank_indicator(
        indicator_code=indicator_code,
        output_column=output_column,
        year=WB_YEAR
    )

NY.GDP.MKTP.CD: 260 country rows, 254 non-null values
SP.POP.TOTL: 260 country rows, 260 non-null values
NY.GDP.PCAP.CD: 260 country rows, 254 non-null values


In [ ]:
wb_gdp = wb_indicator_tables[
    'NY.GDP.MKTP.CD'
].copy()

wb_population = wb_indicator_tables[
    'SP.POP.TOTL'
].copy()

wb_gdp_per_capita = wb_indicator_tables[
    'NY.GDP.PCAP.CD'
].copy()


# Start with the GDP table
wb_2019 = wb_gdp.copy()


# Add population
wb_2019 = wb_2019.merge(
    wb_population[
        [
            'iso_code',
            'wb_population'
        ]
    ],
    on='iso_code',
    how='outer',
    validate='one_to_one'
)


# Add GDP per capita
wb_2019 = wb_2019.merge(
    wb_gdp_per_capita[
        [
            'iso_code',
            'wb_gdp_per_capita_current_usd'
        ]
    ],
    on='iso_code',
    how='outer',
    validate='one_to_one'
)


# Ensure the snapshot year is explicit
wb_2019['wb_year'] = WB_YEAR


print(
    "Prepared World Bank table:",
    wb_2019.shape
)

print(
    "Unique World Bank ISO-3 codes:",
    wb_2019['iso_code'].nunique()
)

display(
    wb_2019.head()
)

Prepared World Bank table: (260, 6)
Unique World Bank ISO-3 codes: 260


,iso_code,wb_country,wb_year,wb_gdp_current_usd,wb_population,wb_gdp_per_capita_current_usd
0,ABW,Aruba,2019,3.347562e+09,109203.0,30654.485124
1,AFE,Africa Eastern and Southern,2019,1.019354e+12,675950189.0,1508.031525
2,AFG,Afghanistan,2019,1.879944e+10,37856121.0,496.602504
3,AFW,Africa Western and Central,2019,1.026528e+12,462522286.0,2219.412555
4,AGO,Angola,2019,8.119381e+10,32375632.0,2507.868072


### 5.4 Validate World Bank Country Keys

Before the World Bank data is merged onto power plants, the prepared country-level table is validated.

The checks confirm:

- ISO-3 keys are unique,
- the table contains no duplicate country records,
- the selected indicators are available,
- and the Philippines is present with the expected 2019 economic and demographic values.

In [ ]:
wb_duplicate_iso = (
    wb_2019
    .duplicated(
        subset=['iso_code']
    )
    .sum()
)

wb_phl = wb_2019[
    wb_2019['iso_code'].eq(
        'PHL'
    )
].copy()

print(
    "World Bank rows:",
    len(wb_2019)
)

print(
    "Unique ISO-3 codes:",
    wb_2019['iso_code'].nunique()
)

print(
    "Duplicate ISO-3 codes:",
    wb_duplicate_iso
)

print(
    "Philippines present in World Bank?:",
    not wb_phl.empty
)

if not wb_phl.empty:

    print(
        "PHL GDP populated?:",
        wb_phl[
            'wb_gdp_current_usd'
        ].notna().all()
    )

    print(
        "PHL population populated?:",
        wb_phl[
            'wb_population'
        ].notna().all()
    )

    print(
        "PHL GDP per capita populated?:",
        wb_phl[
            'wb_gdp_per_capita_current_usd'
        ].notna().all()
    )

    display(
        wb_phl[
            [
                'iso_code',
                'wb_country',
                'wb_year',
                'wb_gdp_current_usd',
                'wb_population',
                'wb_gdp_per_capita_current_usd'
            ]
        ]
    )


assert wb_duplicate_iso == 0, (
    "Duplicate World Bank ISO codes found."
)

assert not wb_phl.empty, (
    "Philippines is missing from "
    "the World Bank table."
)

print(
    "\nWorld Bank country-key "
    "validation passed."
)

World Bank rows: 260
Unique ISO-3 codes: 260
Duplicate ISO-3 codes: 0
Philippines present in World Bank?: True
PHL GDP populated?: True
PHL population populated?: True
PHL GDP per capita populated?: True


,iso_code,wb_country,wb_year,wb_gdp_current_usd,wb_population,wb_gdp_per_capita_current_usd
182,PHL,Philippines,2019,3.768234e+11,110804683.0,3400.789498



World Bank country-key validation passed.


### 5.5 Restrict World Bank Data to GPPD Countries

The World Bank API may return regional and economic aggregates in addition to individual countries.

For the final integration, the World Bank table is restricted to ISO-3 country codes that are present in the Global Power Plant Database.

This prevents non-country World Bank aggregates from entering the plant-level merge while preserving all real power plant records.

In [ ]:
gppd_country_codes = set(
    df_plants['country']
    .dropna()
    .unique()
)

wb_2019_gppd = (
    wb_2019[
        wb_2019['iso_code'].isin(
            gppd_country_codes
        )
    ]
    .copy()
)

print(
    "World Bank rows before GPPD-country filter:",
    len(wb_2019)
)

print(
    "World Bank rows after GPPD-country filter:",
    len(wb_2019_gppd)
)

print(
    "GPPD countries:",
    df_plants['country'].nunique()
)

print(
    "World Bank countries matching GPPD:",
    wb_2019_gppd['iso_code'].nunique()
)

print(
    "Duplicate ISO-3 codes after filter:",
    wb_2019_gppd.duplicated(
        subset=['iso_code']
    ).sum()
)

print(
    "Philippines retained?:",
    'PHL' in set(
        wb_2019_gppd['iso_code']
    )
)

World Bank rows before GPPD-country filter: 260
World Bank rows after GPPD-country filter: 162
GPPD countries: 167
World Bank countries matching GPPD: 162
Duplicate ISO-3 codes after filter: 0
Philippines retained?: True


### 5.6 World Bank Coverage Against GPPD

The prepared World Bank table is compared with the country codes present in the Global Power Plant Database.

Countries without a World Bank match are documented before the final merge. Their power plant records will still be retained through a left join, and unavailable World Bank values will remain missing rather than being replaced with zero.

In [ ]:
gppd_country_lookup = (
    df_plants[
        [
            'country',
            'country_long'
        ]
    ]
    .drop_duplicates()
)

unmatched_wb_countries = (
    gppd_country_lookup[
        ~gppd_country_lookup['country'].isin(
            wb_2019_gppd['iso_code']
        )
    ]
    .sort_values('country')
    .reset_index(drop=True)
)

print(
    "GPPD countries without World Bank match:",
    len(unmatched_wb_countries)
)

display(unmatched_wb_countries)

GPPD countries without World Bank match: 5


,country,country_long
0,ATA,Antarctica
1,ESH,Western Sahara
2,GUF,French Guiana
3,KOS,Kosovo
4,TWN,Taiwan


## 6. Main Three-Source Plant-Level Merge

The cleaned Global Power Plant Database remains the base table.

Country-level installed-capacity features, 2019 OWID emissions indicators, and 2019 World Bank economic and demographic indicators are merged onto the individual power plant records.

All country-level joins use ISO-3 country codes and preserve the natural plant-level grain:

**one row = one real power plant**

Left joins are used so that power plants are not removed when a country-level source does not contain a matching record.

In [ ]:
# Country + primary-fuel features that correspond
# to each plant's own primary fuel
plant_fuel_features = (
    country_fuel_capacity[
        [
            'country',
            'primary_fuel',
            'plant_count',
            'capacity_mw',
            'capacity_share_pct'
        ]
    ]
    .rename(
        columns={
            'plant_count':
                'country_primary_fuel_plant_count',
            'capacity_mw':
                'country_primary_fuel_capacity_mw',
            'capacity_share_pct':
                'country_primary_fuel_capacity_share_pct'
        }
    )
)

# Start from the real plant-level table
eta_merged_2019 = df_plants.copy()

rows_before_feature_merge = len(
    eta_merged_2019
)

# Add country-wide capacity features
eta_merged_2019 = eta_merged_2019.merge(
    country_capacity_features,
    on='country',
    how='left',
    validate='many_to_one'
)

# Add features corresponding to each plant's
# own primary fuel
eta_merged_2019 = eta_merged_2019.merge(
    plant_fuel_features,
    on=[
        'country',
        'primary_fuel'
    ],
    how='left',
    validate='many_to_one'
)

print(
    "Rows before capacity-feature merge:",
    rows_before_feature_merge
)

print(
    "Rows after capacity-feature merge:",
    len(eta_merged_2019)
)

Rows before capacity-feature merge: 34936
Rows after capacity-feature merge: 34936


In [ ]:
rows_before_owid_merge = len(
    eta_merged_2019
)

eta_merged_2019 = eta_merged_2019.merge(
    df_co2_2019,
    left_on='country',
    right_on='iso_code',
    how='left',
    validate='many_to_one',
    indicator='_owid_merge'
)

# Keep a simple data-quality flag
eta_merged_2019['owid_2019_matched'] = (
    eta_merged_2019['_owid_merge']
    .eq('both')
)

print(
    "Rows before OWID merge:",
    rows_before_owid_merge
)

print(
    "Rows after OWID merge:",
    len(eta_merged_2019)
)

print(
    "Countries in final plant-level dataset:",
    eta_merged_2019['country'].nunique()
)

print(
    "Philippines present?:",
    'PHL' in set(
        eta_merged_2019['country']
    )
)

Rows before OWID merge: 34936
Rows after OWID merge: 34936
Countries in final plant-level dataset: 167
Philippines present?: True


### 6.1 Merge World Bank 2019 Indicators

The prepared World Bank country table is merged onto the existing plant-level WRI + OWID dataset.

The relationship is:

**many power plants → one World Bank country record**

The merge uses a left join with `validate='many_to_one'` so the number of real power plant rows must remain unchanged.

In [ ]:
# Rename the World Bank ISO field so its source
# remains explicit after the merge
wb_2019_merge = (
    wb_2019_gppd
    .rename(
        columns={
            'iso_code': 'wb_iso_code'
        }
    )
    .copy()
)

rows_before_wb_merge = len(
    eta_merged_2019
)

unique_plants_before_wb_merge = (
    eta_merged_2019['gppd_idnr']
    .nunique()
)

eta_merged_2019 = eta_merged_2019.merge(
    wb_2019_merge,
    left_on='country',
    right_on='wb_iso_code',
    how='left',
    validate='many_to_one',
    indicator='_wb_merge'
)

# Simple World Bank match flag
eta_merged_2019['wb_2019_matched'] = (
    eta_merged_2019['_wb_merge']
    .eq('both')
)

rows_after_wb_merge = len(
    eta_merged_2019
)

unique_plants_after_wb_merge = (
    eta_merged_2019['gppd_idnr']
    .nunique()
)

print(
    "Rows before World Bank merge:",
    rows_before_wb_merge
)

print(
    "Rows after World Bank merge:",
    rows_after_wb_merge
)

print(
    "Unique plants before World Bank merge:",
    unique_plants_before_wb_merge
)

print(
    "Unique plants after World Bank merge:",
    unique_plants_after_wb_merge
)

print(
    "World Bank matched plant rows:",
    eta_merged_2019['wb_2019_matched'].sum()
)

print(
    "World Bank unmatched plant rows:",
    (~eta_merged_2019['wb_2019_matched']).sum()
)

print(
    "Philippines present after World Bank merge?:",
    'PHL' in set(
        eta_merged_2019['country']
    )
)

Rows before World Bank merge: 34936
Rows after World Bank merge: 34936
Unique plants before World Bank merge: 34936
Unique plants after World Bank merge: 34936
World Bank matched plant rows: 34886
World Bank unmatched plant rows: 50
Philippines present after World Bank merge?: True


In [ ]:
assert (
    rows_after_wb_merge
    ==
    rows_before_wb_merge
), "World Bank merge changed the number of plant rows."

assert (
    unique_plants_after_wb_merge
    ==
    unique_plants_before_wb_merge
), "World Bank merge changed the number of unique plants."

assert (
    eta_merged_2019['gppd_idnr']
    .duplicated()
    .sum()
    == 0
), "World Bank merge introduced duplicate plant IDs."

assert 'PHL' in set(
    eta_merged_2019['country']
), "Philippines was lost during the World Bank merge."

print(
    "World Bank plant-level merge integrity checks passed."
)

World Bank plant-level merge integrity checks passed.


## 7. Final Validation

The completed three-source dataset is validated to confirm that the natural plant-level grain has been preserved after integrating:

- Global Power Plant Database plant attributes,
- OWID 2019 emissions indicators,
- World Bank 2019 economic and demographic indicators,
- and engineered country-level installed-capacity features.

The checks verify row integrity, plant identifiers, source-match coverage, unmatched country codes, and retention of Philippine power plants.

In [ ]:
final_rows = len(
    eta_merged_2019
)

unique_final_plant_ids = (
    eta_merged_2019['gppd_idnr']
    .nunique()
)

duplicate_final_plant_ids = (
    eta_merged_2019['gppd_idnr']
    .duplicated()
    .sum()
)

philippine_final_rows = (
    eta_merged_2019['country']
    .eq('PHL')
    .sum()
)

owid_matched_plant_rows = (
    eta_merged_2019['owid_2019_matched']
    .sum()
)

owid_unmatched_plant_rows = (
    (~eta_merged_2019['owid_2019_matched'])
    .sum()
)

wb_matched_plant_rows = (
    eta_merged_2019['wb_2019_matched']
    .sum()
)

wb_unmatched_plant_rows = (
    (~eta_merged_2019['wb_2019_matched'])
    .sum()
)


# Unmatched OWID countries
unmatched_owid_countries = (
    eta_merged_2019.loc[
        ~eta_merged_2019['owid_2019_matched'],
        [
            'country',
            'country_long'
        ]
    ]
    .value_counts()
    .reset_index(
        name='plant_rows'
    )
    .sort_values('country')
)


# Unmatched World Bank countries
unmatched_wb_countries_final = (
    eta_merged_2019.loc[
        ~eta_merged_2019['wb_2019_matched'],
        [
            'country',
            'country_long'
        ]
    ]
    .value_counts()
    .reset_index(
        name='plant_rows'
    )
    .sort_values('country')
)


# Verify plant identities were preserved
same_plant_ids = (
    set(df_plants['gppd_idnr'])
    ==
    set(eta_merged_2019['gppd_idnr'])
)


print("Original source rows:", len(df_power))
print("Clean plant rows:", len(df_plants))
print("Final merged rows:", final_rows)

print(
    "Unique final plant IDs:",
    unique_final_plant_ids
)

print(
    "Duplicate final plant IDs:",
    duplicate_final_plant_ids
)

print(
    "Countries in final dataset:",
    eta_merged_2019['country'].nunique()
)

print(
    "Philippines present?:",
    'PHL' in set(
        eta_merged_2019['country']
    )
)

print(
    "Philippine plant rows:",
    philippine_final_rows
)

print(
    "\nOWID matched plant rows:",
    owid_matched_plant_rows
)

print(
    "OWID unmatched plant rows:",
    owid_unmatched_plant_rows
)

print(
    "\nWorld Bank matched plant rows:",
    wb_matched_plant_rows
)

print(
    "World Bank unmatched plant rows:",
    wb_unmatched_plant_rows
)

print(
    "\nPlant IDs preserved exactly?:",
    same_plant_ids
)


print("\nUnmatched OWID country ISO codes:")
display(
    unmatched_owid_countries
)

print("\nUnmatched World Bank country ISO codes:")
display(
    unmatched_wb_countries_final
)


# Hard validation checks
assert final_rows == len(df_plants), (
    "Final row count does not match the cleaned "
    "plant-level dataset."
)

assert unique_final_plant_ids == len(df_plants), (
    "Unique plant count changed during integration."
)

assert duplicate_final_plant_ids == 0, (
    "Duplicate plant IDs were introduced."
)

assert same_plant_ids, (
    "Plant identities changed during integration."
)

assert 'PHL' in set(
    eta_merged_2019['country']
), (
    "Philippines was lost from the final dataset."
)

assert philippine_final_rows == 123, (
    "Unexpected number of Philippine plant rows."
)

assert (
    wb_2019_gppd['iso_code']
    .duplicated()
    .sum()
    == 0
), (
    "Duplicate World Bank ISO-3 keys detected."
)

print(
    "\nAll three-source plant-level "
    "integrity checks passed."
)

Original source rows: 34936
Clean plant rows: 34936
Final merged rows: 34936
Unique final plant IDs: 34936
Duplicate final plant IDs: 0
Countries in final dataset: 167
Philippines present?: True
Philippine plant rows: 123

OWID matched plant rows: 34927
OWID unmatched plant rows: 9

World Bank matched plant rows: 34886
World Bank unmatched plant rows: 50

Plant IDs preserved exactly?: True

Unmatched OWID country ISO codes:


,country,country_long,plant_rows
2,ESH,Western Sahara,1
0,GUF,French Guiana,6
1,KOS,Kosovo,2



Unmatched World Bank country ISO codes:


,country,country_long,plant_rows
2,ATA,Antarctica,2
4,ESH,Western Sahara,1
1,GUF,French Guiana,6
3,KOS,Kosovo,2
0,TWN,Taiwan,39



All three-source plant-level integrity checks passed.


### 7.1 Philippines Three-Source Validation

The Philippine plant records are inspected to confirm that they retain:

- WRI plant-level attributes,
- OWID emissions indicators,
- World Bank economic and demographic indicators,
- and engineered installed-capacity features.

This is a merge validation only and is not yet the final Philippine analysis.

In [ ]:
phl_final = (
    eta_merged_2019[
        eta_merged_2019['country']
        .eq('PHL')
    ]
    .copy()
)

print(
    "Philippine plant rows:",
    len(phl_final)
)

print(
    "All Philippine rows matched OWID?:",
    phl_final[
        'owid_2019_matched'
    ].all()
)

print(
    "All Philippine rows matched World Bank?:",
    phl_final[
        'wb_2019_matched'
    ].all()
)

print(
    "PHL World Bank GDP populated?:",
    phl_final[
        'wb_gdp_current_usd'
    ].notna().all()
)

print(
    "PHL World Bank population populated?:",
    phl_final[
        'wb_population'
    ].notna().all()
)

print(
    "PHL World Bank GDP per capita populated?:",
    phl_final[
        'wb_gdp_per_capita_current_usd'
    ].notna().all()
)


phl_preview_columns = [
    'country',
    'country_long',
    'name',
    'gppd_idnr',
    'capacity_mw',
    'primary_fuel',
    'latitude',
    'longitude',
    'commissioning_year',
    'total_installed_capacity_mw',
    'country_primary_fuel_capacity_mw',
    'country_primary_fuel_capacity_share_pct',
    'fossil_share_pct',
    'renewable_share_pct',
    'co2',
    'co2_per_capita',
    'wb_gdp_current_usd',
    'wb_population',
    'wb_gdp_per_capita_current_usd'
]

phl_preview_columns = [
    col
    for col in phl_preview_columns
    if col in phl_final.columns
]

display(
    phl_final[
        phl_preview_columns
    ].head(10)
)

Philippine plant rows: 123
All Philippine rows matched OWID?: True
All Philippine rows matched World Bank?: True
PHL World Bank GDP populated?: True
PHL World Bank population populated?: True
PHL World Bank GDP per capita populated?: True


,country,country_long,name,gppd_idnr,capacity_mw,primary_fuel,latitude,longitude,commissioning_year,total_installed_capacity_mw,country_primary_fuel_capacity_mw,country_primary_fuel_capacity_share_pct,fossil_share_pct,renewable_share_pct,co2,co2_per_capita,wb_gdp_current_usd,wb_population,wb_gdp_per_capita_current_usd
18314,PHL,Philippines,AGUS 1,WRI1001957,80.0,Hydro,8.0046,124.2868,NaN,20719.3,3401.1,16.415130,68.527412,31.472588,142.601,1.287,3.768234e+11,110804683.0,3400.789498
18315,PHL,Philippines,AGUS 2,WRI1001958,180.0,Hydro,8.0528,124.2709,NaN,20719.3,3401.1,16.415130,68.527412,31.472588,142.601,1.287,3.768234e+11,110804683.0,3400.789498
18316,PHL,Philippines,AGUS 4,WRI1001959,158.1,Hydro,8.1336,124.1983,NaN,20719.3,3401.1,16.415130,68.527412,31.472588,142.601,1.287,3.768234e+11,110804683.0,3400.789498
18317,PHL,Philippines,AGUS 5,WRI1001960,309.0,Hydro,8.1937,124.1915,NaN,20719.3,3401.1,16.415130,68.527412,31.472588,142.601,1.287,3.768234e+11,110804683.0,3400.789498
18318,PHL,Philippines,AMBUKLAO,WRI1001902,105.0,Hydro,16.4588,120.7452,NaN,20719.3,3401.1,16.415130,68.527412,31.472588,142.601,1.287,3.768234e+11,110804683.0,3400.789498
18319,PHL,Philippines,ANGAT,WRI1001900,246.0,Hydro,14.9098,121.1603,NaN,20719.3,3401.1,16.415130,68.527412,31.472588,142.601,1.287,3.768234e+11,110804683.0,3400.789498
18320,PHL,Philippines,Anda,WRI1029959,83.7,Coal,15.2342,120.6109,2016.0,20719.3,8731.3,42.140902,68.527412,31.472588,142.601,1.287,3.768234e+11,110804683.0,3400.789498
18321,PHL,Philippines,Armenia Solar,WRI1029964,8.8,Solar,15.4605,120.5918,2016.0,20719.3,1196.4,5.774326,68.527412,31.472588,142.601,1.287,3.768234e+11,110804683.0,3400.789498
18322,PHL,Philippines,Avion,WRI1029960,100.0,Gas,13.7606,121.0460,2016.0,20719.3,3411.0,16.462911,68.527412,31.472588,142.601,1.287,3.768234e+11,110804683.0,3400.789498
18323,PHL,Philippines,BACMAN,WRI1001889,130.0,Geothermal,13.0547,123.9678,NaN,20719.3,1848.2,8.920186,68.527412,31.472588,142.601,1.287,3.768234e+11,110804683.0,3400.789498


## 8. Export Final Dataset

Only one final plant-level dataset is exported.

The completed dataset combines:

- WRI plant-level power plant attributes,
- engineered country-level installed-capacity features,
- OWID 2019 emissions and carbon indicators,
- and World Bank 2019 economic and demographic indicators.

The natural plant-level grain is preserved:

**one row = one real power plant**

Final output:

**`eta_merged_2019.csv`**

In [ ]:
# Create the final export copy while keeping
# temporary merge indicators available in memory
# for notebook validation.
eta_merged_2019_export = (
    eta_merged_2019
    .drop(
        columns=[
            '_owid_merge',
            '_wb_merge'
        ],
        errors='ignore'
    )
    .copy()
)


# Final export checks
print(
    "Final export shape:",
    eta_merged_2019_export.shape
)

print(
    "Final export rows:",
    len(eta_merged_2019_export)
)

print(
    "Final unique plant IDs:",
    eta_merged_2019_export[
        'gppd_idnr'
    ].nunique()
)

print(
    "Duplicate plant IDs:",
    eta_merged_2019_export[
        'gppd_idnr'
    ]
    .duplicated()
    .sum()
)

print(
    "Philippines present?:",
    'PHL' in set(
        eta_merged_2019_export['country']
    )
)

print(
    "World Bank GDP column present?:",
    'wb_gdp_current_usd'
    in eta_merged_2019_export.columns
)

print(
    "World Bank population column present?:",
    'wb_population'
    in eta_merged_2019_export.columns
)

print(
    "World Bank GDP per capita column present?:",
    'wb_gdp_per_capita_current_usd'
    in eta_merged_2019_export.columns
)


assert (
    len(eta_merged_2019_export)
    == 34936
)

assert (
    eta_merged_2019_export[
        'gppd_idnr'
    ].nunique()
    == 34936
)

assert (
    eta_merged_2019_export[
        'gppd_idnr'
    ]
    .duplicated()
    .sum()
    == 0
)


# Save only the final integrated CSV
eta_merged_2019_export.to_csv(
    'eta_merged_2019.csv',
    index=False
)

print(
    "\nFinal dataset saved successfully."
)

print(
    "Output file: eta_merged_2019.csv"
)


from google.colab import files

files.download(
    'eta_merged_2019.csv'
)

Final export shape: (34936, 162)
Final export rows: 34936
Final unique plant IDs: 34936
Duplicate plant IDs: 0
Philippines present?: True
World Bank GDP column present?: True
World Bank population column present?: True
World Bank GDP per capita column present?: True

Final dataset saved successfully.
Output file: eta_merged_2019.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Appendix — Optional Observed Generation Coverage

The GPPD also contains annual `generation_gwh` fields for 2013–2019.

These values are not used as a requirement for inclusion in the final plant-level dataset because their geographic coverage is limited.

Missing `generation_gwh` values remain missing and are never converted to zero.

This appendix is exploratory only and does not create another output CSV.

In [ ]:
generation_cols = [
    col
    for col in df_power.columns
    if col.startswith(
        'generation_gwh_'
    )
]

df_generation_long = df_power.melt(
    id_vars=[
        'country',
        'gppd_idnr'
    ],
    value_vars=generation_cols,
    var_name='generation_year',
    value_name='generation_gwh'
)

df_generation_long['year'] = (
    df_generation_long[
        'generation_year'
    ]
    .str.extract(
        r'(\d{4})'
    )
    .astype(int)
)

# Keep only genuinely observed generation values
df_generation_observed = (
    df_generation_long
    .dropna(
        subset=['generation_gwh']
    )
    .copy()
)

generation_country_coverage = (
    df_generation_observed
    .groupby('year')['country']
    .nunique()
)

print(
    "Countries with observed generation by year:"
)

print(
    generation_country_coverage
)

print(
    "\nPhilippines has observed plant-level generation in the "
    "GPPD generation_gwh fields?:",
    'PHL' in set(
        df_generation_observed['country']
    )
)

print(
    "Note: False indicates a source-data coverage limitation. "
    "Philippine plants are still retained in the final plant-level dataset."
)

Countries with observed generation by year:
year
2013     5
2014     5
2015    29
2016    30
2017    29
2018     4
2019     1
Name: country, dtype: int64

Philippines has observed plant-level generation in the GPPD generation_gwh fields?: False
Note: False indicates a source-data coverage limitation. Philippine plants are still retained in the final plant-level dataset.
